In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from datetime import datetime

# Creating Spark Session
spark = SparkSession.builder.appName("Spark with Hive").enableHiveSupport().getOrCreate()

# HDFS path
ad_campaigns_data_hdfs_path = "/assignment_1_data/ad_campaigns_data.json"
store_data_hdfs_path = "/assignment_1_data/store_data.json"
user_profile_data_hdfs_path = "/assignment_1_data/user_profile_data.json"


26/10/17 19:55:06 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [76]:
# creating ad_campaigns schema
ad_campaigns_schema = """
                campaign_country STRING,
                campaign_id STRING,
                campaign_name STRING,
                device_type STRING,
                event_time TIMESTAMP,
                event_type STRING,
                os_type STRING,
                place_id STRING,
                user_id STRING
            """
# creating ad_campaigns_df data frame
ad_campaigns_df = spark.read.format('json')\
        .schema(ad_campaigns_schema)\
        .option("multiLine", "true")\
        .load(ad_campaigns_data_hdfs_path)

ad_campaigns_df.printSchema()

ad_campaigns_df.show(truncate=False)

root
 |-- campaign_country: string (nullable = true)
 |-- campaign_id: string (nullable = true)
 |-- campaign_name: string (nullable = true)
 |-- device_type: string (nullable = true)
 |-- event_time: timestamp (nullable = true)
 |-- event_type: string (nullable = true)
 |-- os_type: string (nullable = true)
 |-- place_id: string (nullable = true)
 |-- user_id: string (nullable = true)



+----------------+-----------+-----------------------------+-----------+-------------------+----------+-------+---------+-------------------+
|campaign_country|campaign_id|campaign_name                |device_type|event_time         |event_type|os_type|place_id |user_id            |
+----------------+-----------+-----------------------------+-----------+-------------------+----------+-------+---------+-------------------+
|USA             |ABCDFAE    |Food category target campaign|apple      |2018-10-12 13:10:05|impression|ios    |CASSBB-11|1264374214654454321|
|USA             |ABCDFAE    |Food category target campaign|MOTOROLA   |2018-10-12 13:09:04|impression|android|CADGBD-13|1674374214654454321|
|USA             |ABCDFAE    |Food category target campaign|SAMSUNG    |2018-10-12 13:10:10|video ad  |android|BADGBA-12|5747421465445443   |
|USA             |ABCDFAE    |Food category target campaign|SAMSUNG    |2018-10-12 13:10:12|click     |android|CASSBB-11|1864374214654454132|
|USA  

In [77]:
# creating store_schema schema
store_schema = """
                store_name STRING,
                place_ids ARRAY<STRING>
            """
# creating store_df data frame
store_df = spark.read.format('json')\
        .schema(store_schema)\
        .option("multiLine", "true")\
        .load(store_data_hdfs_path)

store_df.printSchema()

store_df.show(truncate=False)

root
 |-- store_name: string (nullable = true)
 |-- place_ids: array (nullable = true)
 |    |-- element: string (containsNull = true)



+--------------+---------------------------------+
|store_name    |place_ids                        |
+--------------+---------------------------------+
|McDonald      |[CASSBB-11, CADGBD-13, FDBEGD-14]|
|BurgerKing    |[CASSBB-11, BADGBA-12]           |
|Macys         |[BADGBA-13, CASSBB-15, FDBEGD-15]|
|shoppers stop |[BADGBA-12, CADGBD-13]           |
|Subway        |[CADGBD-13, FDBEGD-14, BADGBA-13]|
|Walmart       |[CASSBB-11, FDBEGD-15, BADGBA-12]|
|Target        |[CASSBB-15, CADGBD-13]           |
|Starbucks     |[FDBEGD-14, BADGBA-13, CASSBB-11]|
|KFC           |[BADGBA-12, FDBEGD-15]           |
|Dominos       |[CADGBD-13, CASSBB-15, FDBEGD-14]|
|Pizza Hut     |[BADGBA-13, CASSBB-11]           |
|Taco Bell     |[FDBEGD-15, BADGBA-12, CADGBD-13]|
|Costco        |[CASSBB-15, FDBEGD-14]           |
|Best Buy      |[BADGBA-13, FDBEGD-15]           |
|Nike          |[CASSBB-11, BADGBA-12, CASSBB-15]|
|Adidas        |[CADGBD-13, FDBEGD-14]           |
|Apple Store   |[FDBEGD-15, CAS

In [78]:
# creating user_profile_schema schema
user_profile_schema = """
                user_id STRING,
                country STRING,
                gender STRING,
                age_group STRING,
                category ARRAY<STRING>
            """
# creating user_profile_df data frame
user_profile_df = spark.read.format('json')\
        .schema(user_profile_schema)\
        .option("multiLine", "true")\
        .load(user_profile_data_hdfs_path)

user_profile_df.printSchema()

user_profile_df.show(truncate=False)

root
 |-- user_id: string (nullable = true)
 |-- country: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- age_group: string (nullable = true)
 |-- category: array (nullable = true)
 |    |-- element: string (containsNull = true)



+-------------------+-------+------+---------+-------------------------------+
|user_id            |country|gender|age_group|category                       |
+-------------------+-------+------+---------+-------------------------------+
|1264374214654454321|USA    |male  |18-25    |[shopper, student]             |
|1674374214654454321|USA    |female|25-50    |[parent]                       |
|5747421465445443   |USA    |male  |25-50    |[shopper, parent, professional]|
|1864374214654454132|USA    |male  |50+      |[professional]                 |
|14537421465445443  |USA    |female|18-25    |[shopper, student]             |
|25547421465445443  |USA    |female|50+      |[shopper, professional]        |
|2000000000000000001|USA    |male  |18-25    |[student, gamer]               |
|2000000000000000002|Canada |female|25-50    |[shopper, parent]              |
|2000000000000000003|Canada |male  |25-50    |[professional, traveler]       |
|2000000000000000004|USA    |female|18-25    |[stude

In [79]:
# Data Analysis 1

from pyspark.sql.functions import *

hdfs_output_path = "/assignment_1_data/output_data"

# Transformation 1
ad_campaigns_df_new = ad_campaigns_df.withColumn("event_hour",hour(col("event_time"))).withColumn("event_date",to_date(col("event_time")))

# Transformation 2
ad_campaigns_count_df = ad_campaigns_df_new.groupBy("campaign_id","event_date","event_hour","os_type","event_type")\
                        .agg(count("*").alias("event_count"))

# Transformation 3
ad_campaigns_pivot_df = ad_campaigns_count_df.groupBy("campaign_id","event_date","event_hour","os_type")\
                        .pivot("event_type")\
                        .sum("event_count")\
                        .fillna(0)\
                        .orderBy("campaign_id")

# Transformation 4
ad_campaigns_pivot_df = ad_campaigns_pivot_df.withColumn(
    "event",
    create_map(
    lit("impression"),col("impression"),
    lit("click"),col("click"),
    lit("video ad"),col("video ad")
    )
)

# Transformation 5
ad_campaigns_final_df =  ad_campaigns_pivot_df.withColumnRenamed("event_date","date")\
.withColumnRenamed("event_hour","hour")\
.withColumnRenamed("os_type","value")\
.withColumn("type", lit("os_type"))

# Transformation 6
ad_campaigns_final_df = ad_campaigns_final_df.select("campaign_id","date","hour","type","value","event")
ad_campaigns_final_df.show()

# writing to hdfs path
ad_campaigns_final_df.write.json(hdfs_output_path + "_1", mode="overwrite")

+-----------+----------+----+-------+-------+--------------------+
|campaign_id|      date|hour|   type|  value|               event|
+-----------+----------+----+-------+-------+--------------------+
|    ABCDFAE|2018-10-12|  13|os_type|android|{impression -> 2,...|
|    ABCDFAE|2018-10-12|  13|os_type|    ios|{impression -> 3,...|
|    LMN456A|2018-10-12|  15|os_type|android|{impression -> 3,...|
|    LMN456A|2018-10-12|  15|os_type|    ios|{impression -> 2,...|
|    QWER789|2018-10-13|  10|os_type|    ios|{impression -> 3,...|
|    QWER789|2018-10-13|  10|os_type|android|{impression -> 2,...|
|    XYZPQR1|2018-10-12|  14|os_type|    ios|{impression -> 1,...|
|    XYZPQR1|2018-10-12|  14|os_type|android|{impression -> 4,...|
+-----------+----------+----+-------+-------+--------------------+



In [80]:
# Data Analysis 2

from pyspark.sql.functions import *

hdfs_output_path = "/assignment_1_data/output_data"

# Transformation 1
store_df_exploded = store_df.select("store_name",explode("place_ids").alias("place_id"))

# Transformation 2
joined_df = ad_campaigns_df.join(store_df_exploded, on="place_id", how = "inner")

# Transformation 3
store_df_new = joined_df.withColumn("event_hour",hour(col("event_time"))).withColumn("event_date",to_date(col("event_time")))

# Transformation 4
store_count_df = store_df_new.groupBy("campaign_id","event_date","event_hour","store_name","event_type")\
                        .agg(count("*").alias("event_count"))

# Transformation 5
store_pivot_df = store_count_df.groupBy("campaign_id","event_date","event_hour","store_name")\
                        .pivot("event_type")\
                        .sum("event_count")\
                        .fillna(0)

# Transformation 6
store_pivot_df = store_pivot_df.withColumn(
    "event",
    create_map(
    lit("impression"),col("impression"),
    lit("click"),col("click"),
    lit("video ad"),col("video ad")
    )
)

# Transformation 7
store_final_df =  store_pivot_df.withColumnRenamed("event_date","date")\
.withColumnRenamed("event_hour","hour")\
.withColumnRenamed("store_name","value")\
.withColumn("type", lit("store_name"))

# Transformation 8
store_final_df = store_final_df.select("campaign_id","date","hour","type","value","event")
store_final_df.show()

# Writing to HDFS path
store_final_df.write.json(hdfs_output_path + "_2", mode="overwrite")


26/10/17 22:03:41 INFO PlanChangeLogger: 
 Dataproc Rule org.apache.spark.sql.catalyst.optimizer.google.ConvertToLeftSemiJoin effective 1 times.



+-----------+----------+----+----------+--------------+--------------------+
|campaign_id|      date|hour|      type|         value|               event|
+-----------+----------+----+----------+--------------+--------------------+
|    XYZPQR1|2018-10-12|  14|store_name|Dominos Canada|{impression -> 2,...|
|    QWER789|2018-10-13|  10|store_name|           KFC|{impression -> 2,...|
|    LMN456A|2018-10-12|  15|store_name|        Subway|{impression -> 1,...|
|    ABCDFAE|2018-10-12|  13|store_name|   Tim Hortons|{impression -> 2,...|
|    QWER789|2018-10-13|  10|store_name|          Nike|{impression -> 2,...|
|    XYZPQR1|2018-10-12|  14|store_name|      McDonald|{impression -> 4,...|
|    LMN456A|2018-10-12|  15|store_name| shoppers stop|{impression -> 2,...|
|    ABCDFAE|2018-10-12|  13|store_name|     Starbucks|{impression -> 3,...|
|    QWER789|2018-10-13|  10|store_name|         Macys|{impression -> 1,...|
|    ABCDFAE|2018-10-12|  13|store_name|           KFC|{impression -> 0,...|

In [81]:
# Data Analysis 3

from pyspark.sql.functions import *

hdfs_output_path = "/assignment_1_data/output_data"

# Transformation 1
join_df = user_profile_df.select("gender","user_id").join(ad_campaigns_df, on="user_id", how = "inner")

# Transformation 2
user_profile_new_df = join_df.withColumn("event_hour",hour(col("event_time"))).withColumn("event_date",to_date(col("event_time")))

# Transformation 3
user_profile_count_df = user_profile_new_df.groupBy("campaign_id","event_date","event_hour","gender","event_type")\
                        .agg(count("*").alias("event_count"))

# Transformation 4
user_profile_pivot_df = user_profile_count_df.groupBy("campaign_id","event_date","event_hour","gender")\
                        .pivot("event_type")\
                        .sum("event_count")\
                        .fillna(0)\
                        .withColumn(
                        "event",
                        create_map(
                        lit("impression"),col("impression"),
                        lit("click"),col("click"),
                        lit("video ad"),col("video ad")
                        )
                    )

# Transformation 5
user_profile_final_df =  user_profile_pivot_df.withColumnRenamed("event_date","date")\
.withColumnRenamed("event_hour","hour")\
.withColumnRenamed("gender","value")\
.withColumn("type", lit("gender"))

# Transformation 6
user_profile_final_df = user_profile_final_df.select("campaign_id","date","hour","type","value","event")
user_profile_final_df.show()

# Writing to HDFS path
store_final_df.write.json(hdfs_output_path + "_3", mode="overwrite")


26/10/17 22:04:19 INFO PlanChangeLogger: 
 Dataproc Rule org.apache.spark.sql.catalyst.optimizer.google.ConvertToLeftSemiJoin effective 1 times.



+-----------+----------+----+------+------+--------------------+
|campaign_id|      date|hour|  type| value|               event|
+-----------+----------+----+------+------+--------------------+
|    XYZPQR1|2018-10-12|  14|gender|  male|{impression -> 2,...|
|    QWER789|2018-10-13|  10|gender|female|{impression -> 3,...|
|    QWER789|2018-10-13|  10|gender|  male|{impression -> 2,...|
|    LMN456A|2018-10-12|  15|gender|  male|{impression -> 2,...|
|    ABCDFAE|2018-10-12|  13|gender|  male|{impression -> 2,...|
|    XYZPQR1|2018-10-12|  14|gender|female|{impression -> 3,...|
|    LMN456A|2018-10-12|  15|gender|female|{impression -> 3,...|
|    ABCDFAE|2018-10-12|  13|gender|female|{impression -> 3,...|
+-----------+----------+----+------+------+--------------------+

